# Particle Localization Syntax from U-Track

traj_idx : trajectory index, as identified from linking particles in U-track

frame: the frame index. These are in 20ms intervals if that comes up

x [nm] : x localization in nm
y [nm] : y localization in nm
z [nm] : z localizaiotn in nm

xy_pstd: precision in localization in XY. I'm not sure if the precision is a free parameter in the modeling but we can use empirical values for such things if we want to.

z_pstd: presicions of localization in Z

cd_class: chromatin density class. This probably not relevant for now but might be for next steps. You might consider dropping all of the entries that have cd_class==0 as these are measurements outside of the region of interest.

scram_class: scrambled chromatin density class. probably not relevant for now but might be for next steps.

ds_idx: index for keeping track of datasets

rep: index for keeping track of replicate

cell_idx: index for keeping track of cell


###### 

# Associated File Paths, Specification for 2D/3D, Max Frame Shifts

In [21]:
selected_file = '/Users/john_1john_1/Documents/Legant_Lab/h2b_tracking_csv_for_john240416/ds210418_ctrl_rep1_cell1_240416.csv'
selected_dim = 2
selected_max_fs = 3

# Import Required Libraries

In [24]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math as math

# Particle Displacements
### Takes x,y,z localizations for particles within a cell. The respective particle displacements are calculated for various frame shifts (temporal shifts) in a linear fashion. Euclidean algorithm is utilized for computing either 2D or 3D displacements.

In [27]:
%%writefile "/Users/john_1john_1/Dropbox/Legant Lab/Scripts/DisplacementCalculation_3col.py"
def calculate_displacement_3_column(max_dt,dim,file_path):
    """max_dt --> max frame step, dim --> 2D or 3D displacement, file_path --> file location"""
    
    # Initialize a dictionary to store displacement lists for each dt
    disp_dict = {f'dT = {dt}': [] for dt in range(1, max_dt + 1)}
    
    particle_data = pd.read_csv(file_path)
     
    for trajectory in np.unique(particle_data['traj_idx']): # Iterate through each unique trajectory
        particle_traj = particle_data[particle_data['traj_idx'] == trajectory] #boolean selector

        xarr = np.array(particle_traj['x [nm]']) #x,y,z for p trajectory in each frame
        yarr = np.array(particle_traj['y [nm]'])
        zarr = np.array(particle_traj['z [nm]'])
    
        for dt in range(1, max_dt + 1): #1-5 inclusive
            xlocs0 = xarr[:-dt]
            xlocs1 = xarr[dt:]  #dt frames ahead
            ylocs0 = yarr[:-dt]
            ylocs1 = yarr[dt:]  #dt frames ahead
            zlocs0 = zarr[:-dt] 
            zlocs1 = zarr[dt:]  #dt frames ahead
            
            if dim == 2:
                displacement = np.sqrt((xlocs1-xlocs0)**2 + (ylocs1-ylocs0)**2)
            elif dim == 3:
                displacement = np.sqrt((xlocs1-xlocs0)**2 + (ylocs1-ylocs0)**2 + (zlocs1-zlocs0)**2)
            else:
                return "Please enter a valid Dimension"
            
            # Append displacement values to the corresponding list in the dictionary
            disp_dict[f'dT = {dt}'].extend(displacement.tolist())

    # Find the longest list to equalize lengths
    max_length = max(len(lst) for lst in disp_dict.values())
    
    # Equalize lengths of lists by appending NaN for missing values
    for dt in range(1, max_dt + 1):
        current_length = len(disp_dict[f'dT = {dt}'])
        disp_dict[f'dT = {dt}'] += [np.nan] * (max_length - current_length)
    
    # Convert the dictionary to a DataFrame
    displacement_df = pd.DataFrame(disp_dict)
    
    return displacement_df

Writing /Users/john_1john_1/Dropbox/Legant Lab/Scripts/DisplacementCalculation_3col.py


FileNotFoundError: [Errno 2] No such file or directory: '/Users/john_1john_1/Dropbox/Legant Lab/Scripts/DisplacementCalculation_3col.py'

In [29]:
%%writefile "/Users/john_1john_1/Dropbox/Legant Lab/Scripts/DisplacementCalculation_1col.py"
def calculate_displacement(max_dt,dim,file_path):
    """max_dt --> max frame step, dim --> 2D or 3D displacement, file_path --> file location"""
    
    disp_lst = [] #displacement values
    dt_lst = [] #time interval
    
    particle_data = pd.read_csv(file_path)
     
    for trajectory in np.unique(particle_data['traj_idx']): # Iterate through each unique trajectory
        particle_traj = particle_data[particle_data['traj_idx'] == trajectory] #boolean selector

        xarr = np.array(particle_traj['x [nm]']) #x,y,z for p trajectory in each frame
        yarr = np.array(particle_traj['y [nm]'])
        zarr = np.array(particle_traj['z [nm]'])
    
        for dt in range(1, max_dt + 1): #1-5 inclusive
            xlocs0 = xarr[:-dt]
            xlocs1 = xarr[dt:]  #dt frames ahead
            ylocs0 = yarr[:-dt]
            ylocs1 = yarr[dt:]  #dt frames ahead
            zlocs0 = zarr[:-dt] 
            zlocs1 = zarr[dt:]  #dt frames ahead
            
            if dim == 2:
                displacement = np.sqrt((xlocs1-xlocs0)**2 + (ylocs1-ylocs0)**2)
            elif dim == 3:
                displacement = np.sqrt((xlocs1-xlocs0)**2 + (ylocs1-ylocs0)**2 + (zlocs1-zlocs0)**2)
            else:
                return "Please enter a valid Dimension"
            
            disp_lst.extend (displacement)
            dt_lst. extend ([dt] * len(displacement)) ##Displacement is in nm
    
    return pd.DataFrame({'dt': dt_lst, 'displacement': disp_lst})

Writing /Users/john_1john_1/Dropbox/Legant Lab/Scripts/DisplacementCalculation_1col.py


FileNotFoundError: [Errno 2] No such file or directory: '/Users/john_1john_1/Dropbox/Legant Lab/Scripts/DisplacementCalculation_1col.py'

In [31]:
calculate_displacement_3_column(selected_max_fs,selected_dim,selected_file)

NameError: name 'calculate_displacement_3_column' is not defined

In [33]:
calculate_displacement(selected_max_fs,selected_dim,selected_file)

NameError: name 'calculate_displacement' is not defined

### Check to see the different temporal shifts

In [ ]:
result_df = calculate_displacement(selected_max_fs,selected_dim,selected_file)
print(result_df['dt'].unique())

# Jump Distance Distributions

### From the previously calcuated displacements for various frame shifts, we can now construct histograms for visualization.

## Standard Histograms

In [ ]:
def plt_jump_length_histogram(max_dt,dim,file_path,log_scale):
    """max_dt --> max frame interation, dim --> 2D or 3D displacement, file_path --> file location
    log_scale --> boolean log plot"""
    
    
    displacements = calculate_displacement(max_dt,dim,file_path)
    
    n_cols = int(np.ceil(np.sqrt(max_dt)))
    n_rows = int(np.ceil(max_dt / n_cols))
    
    fig = plt.figure(figsize=(5 * n_cols, 4 * n_rows))
    
    for dt in range(1, max_dt + 1):  # Start from 1 to max_dt inclusive
        ax = fig.add_subplot(n_rows, n_cols, dt)
        subset = displacements[displacements['dt'] == dt]['displacement'] #boolean in displacement column
        plt.hist(subset, bins=20, alpha=0.5, label=f'dt={dt}', edgecolor='black')
        #bin number ~ 1+3.3(num of samples). But there is a different number of displacements for each dt.
        #not sure if we want to use this rule. For now I will just set a constant number of bins
        #bins=int(1+3.3*(math.log(len(displacements))))
    
        plt.xlabel('Displacement (nm)')
        plt.ylabel('Frequency')
        plt.title(f'JDD for dt={dt}')
        plt.legend()
        plt.tight_layout()
        
        if log_scale == True:
            ax.set_yscale('log')  # Set y-axis to logarithmic scale
        
        ax.set_xlim(0, 600) #set constant to observe differences
        
        
    return plt.show()

In [ ]:
test1_histogram = plt_jump_length_histogram(selected_max_fs,selected_dim,selected_file, False);

Skewed histograms

## Cool Density Plot

In [35]:
def plt_jump_length_density(max_dt,dim,file_path):
    """max_dt --> max frame interation, dim --> 2D or 3D displacement, file_path --> file location"""
    
    plt.figure(figsize=(10, 8))
    displacements = calculate_displacement(max_dt,dim,file_path)
    sns.kdeplot(data=displacements,x='displacement',hue='dt',fill=True,common_norm=False,alpha=0.5,palette="crest")
    
    plt.xlabel('Displacement (nm)')
    plt.ylabel('Density')
    plt.title('Jump Length Density Distribution by dt')
    
    return plt.show()

In [37]:
test1_density_plot = plt_jump_length_density(selected_max_fs,selected_dim,selected_file)

NameError: name 'calculate_displacement' is not defined

<Figure size 1000x800 with 0 Axes>

## ECDF's 

quantifies the rate of convergence of the empirical distribution function to the underlying cumulative distribution function.

In [ ]:
%%writefile "/Users/john_1john_1/Dropbox/Legant Lab/JGparticle/Plot_ECDF.py"
def plt_jdd_ecdf(max_dt,dim,file_path):
    """Plots the ECDF's of particle displacements over a range of frame intervals"""
    
    displacements = calculate_displacement(max_dt,dim,file_path)
    
    plt.figure(figsize=(10,8))
    
    for dt in range(1, max_dt + 1):
        subset = displacements[displacements['dt'] == dt]['displacement']
        
        # Sort the subset to prepare for ECDF plotting
        sorted_subset = np.sort(subset)
        
        # Calculate ECDF values
        y_vals = np.arange(1, len(sorted_subset) + 1) / len(sorted_subset)
        
        plt.plot(sorted_subset, y_vals, marker='.', linestyle='none', label=f'dt={dt}')
        plt.xlabel('Displacement (nm)')
        plt.ylabel('ECDF')
        plt.title('Combined ECDFs for Different Frame Intervals')
        plt.legend(title='Frame Interval (dt)', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
    
    return plt.show()

In [ ]:
plt_jdd_ecdf(selected_max_fs,selected_dim,selected_file)

In [ ]:
!zip -r JGparticle.zip JGparticle/

In [ ]:
%%writefile "/Users/john_1john_1/Dropbox/Legant Lab/JGparticle/Compare_Displacments.py"


import numpy as np
import matplotlib.pyplot as plt
import json

def analyze_and_compare_displacements(TARDIS_df, PYTHON_df, max_dt, log_scale=False, save_results=False):
    """
    Analyzes and compares displacement data between TARDIS and PYTHON data frames.

    Parameters:
    TARDIS_df, PYTHON_df : pandas DataFrames containing displacement data.
    max_dt : int, maximum dT value to consider for histograms.
    log_scale : bool, whether to apply logarithmic scale on plots.
    save_results : bool, whether to save histogram and density ratio data to a file.
    """
    results = {}
    density_ratios = {}
    n_cols = int(np.ceil(np.sqrt(max_dt)))
    n_rows = int(np.ceil(max_dt / n_cols))
    
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 6 * n_rows))
    
    if n_rows * n_cols > 1:
        axs = axs.flatten()
    else:
        axs = [axs]
    
    for i in range(max_dt):
        col_name = f'dT = {i + 1}'
        min_val = min(TARDIS_df[col_name].min(), PYTHON_df[col_name].min())
        max_val = max(TARDIS_df[col_name].max(), PYTHON_df[col_name].max())
        bins = np.arange(min_val, max_val + 15, 15)
        
        tardis_counts, _ = np.histogram(TARDIS_df[col_name], bins=bins, density=True)
        python_counts, bin_edges = np.histogram(PYTHON_df[col_name], bins=bins, density=True)
        
        ratio = np.nan_to_num(python_counts / tardis_counts, nan=0.0, posinf=0.0, neginf=0.0)
        bin_midpoints = (bin_edges[:-1] + bin_edges[1:]) / 2
        
        axs[i].plot(bin_midpoints, ratio, marker='o', linestyle='-', label=f'Ratio {col_name}')
        axs[i].set_xlabel('Displacement (nm)')
        axs[i].set_ylabel('PYTHON / TARDIS Density Ratio')
        axs[i].set_title(f'Density Ratio for {col_name}')
        axs[i].legend(loc='upper right')
        axs[i].set_ylim(0, 2)  # Adjusting the y-axis limits
        
        if log_scale:
            axs[i].set_yscale('log')
        
        # Saving results
        results[col_name] = {'bins': bin_edges, 'tardis_counts': tardis_counts, 'python_counts': python_counts}
        density_ratios[col_name] = {'bins': bin_midpoints, 'density_ratio': ratio}
        
    plt.tight_layout()
    plt.show()
    
    if save_results:
        with open('displacement_analysis_results.json', 'w') as f:
            json.dump({'histogram_data': results, 'density_ratios': density_ratios}, f, indent=4)
    
    return results, density_ratios